# 💬 EasyLLM

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlamandaCathartica/EasyLLM/blob/main/GUI_LLM.ipynb)

A chat GUI for running **LLMs up to 13B** on the free Colab **T4** (4-bit / 8-bit / fp16), with a live log of everything the engine prints.

**No setup, no uploads:** pick a **GPU runtime**, run the cell below and open the link it prints. The engine and the GUI are downloaded automatically from GitHub into the Colab session each time. Pick a model (or paste a Hugging Face link) inside the GUI; no other cell is needed.

The GUI is exposed through a free **Cloudflare quick tunnel** (no account needed). The link contains a secret key; anyone with it can use your GPU, so don't share it. Set `USE_TUNNEL = False` to use Colab's built-in port proxy instead. For gated models (Llama etc.) add a Colab secret named `HF_TOKEN`.

In [ ]:
#@title ▶️ Start EasyLLM
import json, os, secrets, subprocess, sys, time, urllib.request

REPO_URL = 'https://github.com/AlamandaCathartica/EasyLLM' #@param {type:"string"}
BRANCH = 'main' #@param {type:"string"}
USE_TUNNEL = True #@param {type:"boolean"}
USE_DRIVE = False #@param {type:"boolean"}
DEMO = False #@param {type:"boolean"}

ROOT = '/content'
REPO_DIR = f'{ROOT}/EasyLLM'                  # code is downloaded here every session
GUI = f'{REPO_DIR}/gui/easyllm_gui.py'
PORT = 7860
URL_FILE = f'{ROOT}/easyllm_url.txt'
LOG_FILE = f'{ROOT}/easyllm_gui.log'
CACHE_DIR = f'{ROOT}/_easyllm_cache'          # cloudflared binary (and model cache when USE_DRIVE is on)
KEY = secrets.token_urlsafe(16) if USE_TUNNEL else ''

# ---- Google Drive (optional): keeps cloudflared and downloaded models between sessions ----
if USE_DRIVE:
  from google.colab import drive
  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
  CACHE_DIR = '/content/drive/MyDrive/EasyLLM/_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# ---- download / update the engine and GUI from GitHub (nothing to upload by hand) ----
subprocess.run(['pkill', '-f', 'easyllm_gui.py'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
  r = subprocess.run(f'git -C {REPO_DIR} fetch --depth 1 origin {BRANCH} && git -C {REPO_DIR} reset --hard FETCH_HEAD',
                     shell=True, capture_output=True, text=True)
else:
  subprocess.run(['rm', '-rf', REPO_DIR], check=False)
  r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, REPO_DIR], capture_output=True, text=True)
if r.returncode != 0:
  print(r.stdout, r.stderr)
  if not os.path.exists(GUI):
    raise SystemExit(f'Could not download {REPO_URL} (branch {BRANCH}). Check the URL/branch and that the repo is public.')
  print('Update failed; using the copy already in this session.')
assert os.path.exists(GUI), f'{GUI} not found in the downloaded repo.'
VERSION = open(f'{REPO_DIR}/VERSION').read().strip() if os.path.exists(f'{REPO_DIR}/VERSION') else 'unknown'
commit = subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip()
print(f'EasyLLM v{VERSION} ready: {commit}')

# ---- install the Python packages the engine needs (a minute the first time) ----
if not DEMO:
  print('Installing dependencies...')
  r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'],
                     capture_output=True, text=True)
  if r.returncode != 0:
    print(r.stdout[-2000:], r.stderr[-2000:])
    raise SystemExit('pip install failed. See the messages above.')

# ---- start the GUI ----
env = dict(os.environ)
try:
  from google.colab import userdata
  env['HF_TOKEN'] = userdata.get('HF_TOKEN')    # optional secret, only needed for gated models
except Exception:
  pass
if USE_DRIVE:
  env['HF_HOME'] = f'{CACHE_DIR}/huggingface'   # models are downloaded once, then reused
if os.path.exists(URL_FILE): os.remove(URL_FILE)
time.sleep(1)

cmd = [sys.executable, GUI, '--port', str(PORT), '--no-browser', '--cache-dir', CACHE_DIR]
if USE_TUNNEL: cmd += ['--tunnel', '--url-file', URL_FILE, '--key', KEY]
if DEMO: cmd.append('--demo')
log = open(LOG_FILE, 'w')
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)

if USE_TUNNEL:
  print('Starting Cloudflare tunnel (first run downloads cloudflared)...')
  for _ in range(90):
    if os.path.exists(URL_FILE) or proc.poll() is not None: break
    time.sleep(1)
  if os.path.exists(URL_FILE):
    LINK = open(URL_FILE).read()
    print('\n✅ Open EasyLLM here (keep this link private, it contains an access key):\n')
    print(LINK)
  else:
    print(open(LOG_FILE).read()[-3000:]); raise SystemExit('Tunnel did not start. See the log above.')
else:
  from google.colab import output
  time.sleep(2)
  output.serve_kernel_port_as_window(PORT)
  output.serve_kernel_port_as_iframe(PORT, height=1000)

# ---- keep this cell running ------------------------------------------------
# Colab may reclaim a runtime that looks idle. The GUI loads models and chats by itself
# (no other cell is needed), so this cell stays alive, showing the link and a heartbeat.
from IPython.display import clear_output
def _status():
  try:
    q = f'?key={KEY}' if KEY else ''
    d = json.load(urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/state{q}', timeout=5))
    s = d['status']
    if d.get('model'): s += f" | {d['model']} ({d['quant']})"
    g = d['gpu']
    if g['total']: s += f" | VRAM {g['used']:.1f}/{g['total']:.1f} GiB"
    return s
  except Exception as e:
    return f'GUI not responding ({e})'
try:
  while proc.poll() is None:
    clear_output(wait=True)
    if USE_TUNNEL: print('EasyLLM link (keep private):\n' + LINK)
    print('\n' + time.strftime('%H:%M:%S'), '-', _status())
    print('\nLeave this cell running. Stopping it does not stop the GUI; everything is controlled from the GUI.')
    time.sleep(20)
  print(f'The GUI server exited. See {LOG_FILE}')
except KeyboardInterrupt:
  print('Cell stopped. The GUI keeps running in the background.')

Closing the browser tab doesn't unload the model. Use **Unload** in the GUI's settings. Server log: open the **log** button in the GUI, or `/content/easyllm_gui.log`.